# Box-Cox como prefiltrado para segmentación de grietas

Experimento para responder a los revisores (Comentarios 1 y 2):

- **Escala**: 4 sets de 50 imágenes de grietas (Ozgenel & Sorguç) según orientación
  — `Vertical`, `Horizontal`, `Left` (oblicuas izq→der) y `Big` (grieta de gran extensión).
- **Incertidumbre**: se reporta **media ± desviación estándar** sobre las 50 imágenes de
  cada set, con **test pareado** (permutación + Wilcoxon) e **IC95% bootstrap** para el
  efecto de Box-Cox (con − sin).

### Nota metodológica
- **ML tradicional (SVM, LightGBM, KNN, LDA, QDA)**: se entrenan y predicen sobre los
  píxeles de *la misma imagen* (no requieren datos externos). Aplicar Box-Cox o no solo
  cambia la imagen de entrada → ambas variantes son consistentes y comparables.
- **Deep Learning (U-Net, DeepLab, FCN)**: aquí *sí* importa el entrenamiento. Comparamos
  **A** (entrenar + testear con originales) vs **C** (entrenar + testear con Box-Cox), cada
  modelo bajo su propio preprocesamiento. Esto evita el sesgo del protocolo "entrenar en
  originales, testear en transformadas" (shift de distribución), que es la razón por la que
  DL aparentaba no mejorar.
- **Sobre el bootstrap**: el problema de independencia aplica a remuestrear *píxeles dentro
  de una imagen* (autocorrelación espacial). Al remuestrear las **50 imágenes** (observaciones
  independientes), el bootstrap y el test pareado son válidos.

> La métrica `Accuracy` es engañosa (la grieta es ~1% de los píxeles); priorizar **IoU** y **Dice/F1**.

## 0. Dependencias
Ejecutar una vez. Si ya están instaladas, se puede omitir.

In [ ]:
# %pip install -q numpy scipy scikit-learn lightgbm opencv-python-headless pandas
# Para la sección opcional de Deep Learning (4):
# %pip install -q torch torchvision segmentation-models-pytorch

## 1. Configuración

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from scipy import stats
warnings.filterwarnings("ignore")

# Raíz del proyecto: carpeta que contiene data/rgb (se busca hacia arriba desde el cwd)
_HERE = Path.cwd()
ROOT = next((p for p in [_HERE, *_HERE.parents] if (p / "data" / "rgb").exists()),
            Path("/Users/sebastianvidalsalado/Desktop/Crack_Seg"))
RGB_DIR = ROOT / "data" / "rgb"
MASK_DIR = ROOT / "data" / "masks"
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
SETS = {"Vertical": "Vertical", "Horizontal": "Horizontal", "Left": "Left", "Big": "Big"}

SIZE = 256          # se redimensiona a SIZE x SIZE (full-res es inviable por imagen)
TRAIN_CAP = 6000    # máx. píxeles de entrenamiento (estratificados) por imagen
SEED = 42
METRICS = ["IoU", "Dice", "Precision", "Recall", "Accuracy"]
SET_ORDER = ["Vertical", "Horizontal", "Left", "Big"]
METHOD_ORDER = ["SVM", "LightGBM", "KNN", "LDA", "QDA"]
print("ROOT =", ROOT, "| existe rgb:", RGB_DIR.exists())

## 2. Prefiltrado Box-Cox

`color → gris (0.299R+0.587G+0.114B) → Box-Cox (λ por máxima verosimilitud) → histogram
stretching a [0,1]`. Replica `box_cox_img.py` y la Ec. (7) del paper. La condición *sin*
Box-Cox usa la intensidad gris cruda (gris/255). Así la comparación aísla el efecto de la
transformación.

In [ ]:
def to_gray(rgb):
    rgb = rgb.astype(np.float64)
    return 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]

def boxcox_stretch(gray):
    x = gray.flatten().astype(np.float64)
    rng = x.max() - x.min()
    if rng == 0:
        return np.zeros_like(gray), np.nan
    xn = (x - x.min()) / rng + 1e-5            # estrictamente positivo
    bc, lam = stats.boxcox(xn)                  # λ por MLE
    brng = bc.max() - bc.min()
    bc = (bc - bc.min()) / brng if brng > 0 else np.zeros_like(bc)  # histogram stretch
    return bc.reshape(gray.shape), float(lam)

def load_pair(rgb_path, mask_path, size=SIZE):
    rgb = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.uint8)        # 1 = grieta
    return rgb, mask

def make_feature(rgb, use_boxcox):
    gray = to_gray(rgb)
    if use_boxcox:
        return boxcox_stretch(gray)
    return gray / 255.0, np.nan

### Visualización rápida (opcional)

In [ ]:
import matplotlib.pyplot as plt
ex = sorted(p.name for p in (MASK_DIR / "Vertical").glob("*.jpg"))[0]
rgb, mask = load_pair(RGB_DIR / ex, MASK_DIR / "Vertical" / ex)
g_no, _ = make_feature(rgb, False)
g_bc, lam = make_feature(rgb, True)
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, im, t in zip(ax, [rgb, g_no, g_bc, mask],
                    ["RGB", "Gris (sin BC)", f"Box-Cox (λ={lam:.2f})", "Máscara"]):
    a.imshow(im, cmap=None if im.ndim == 3 else "gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 3. Pipeline ML por imagen

Para cada imagen y condición: feature = intensidad (gris o Box-Cox), se ajusta cada
clasificador sobre una submuestra estratificada de píxeles y se predice la imagen completa.
Métricas calculadas para la clase grieta.

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
import lightgbm as lgb

def build_models():
    return {
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", cache_size=500),
        "LightGBM": lgb.LGBMClassifier(n_estimators=100, num_leaves=31, verbose=-1),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "LDA": LinearDiscriminantAnalysis(),
        "QDA": QuadraticDiscriminantAnalysis(reg_param=0.01),
    }

def metrics(y_true, y_pred):
    yt, yp = y_true.astype(bool), y_pred.astype(bool)
    tp = np.sum(yt & yp); fp = np.sum(~yt & yp); fn = np.sum(yt & ~yp); tn = np.sum(~yt & ~yp)
    union = tp + fp + fn
    return dict(
        IoU=tp / union if union > 0 else np.nan,
        Dice=2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else np.nan,
        Precision=tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        Recall=tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        Accuracy=(tp + tn) / (tp + tn + fp + fn),
    )

def stratified_train_idx(y, cap, rng):
    idx0, idx1 = np.where(y == 0)[0], np.where(y == 1)[0]
    n = cap // 2
    return np.concatenate([rng.choice(idx0, min(n, len(idx0)), replace=False),
                           rng.choice(idx1, min(n, len(idx1)), replace=False)])

def run_ml(quick=False):
    rng = np.random.default_rng(SEED)
    rows = []
    for set_name, folder in SETS.items():
        files = sorted(f for f in os.listdir(MASK_DIR / folder) if f.lower().endswith(".jpg"))
        if quick:
            files = files[:3]
        for fi, fname in enumerate(files):
            rgb_path = RGB_DIR / fname
            if not rgb_path.exists():
                continue
            rgb, mask = load_pair(rgb_path, MASK_DIR / folder / fname)
            y = mask.reshape(-1)
            if y.sum() == 0:
                continue
            tr_idx = stratified_train_idx(y, TRAIN_CAP, rng)
            for cond in ("noBC", "BC"):
                feat, lam = make_feature(rgb, use_boxcox=(cond == "BC"))
                X = feat.reshape(-1, 1)
                Xtr, ytr = X[tr_idx], y[tr_idx]
                for mname, model in build_models().items():
                    try:
                        model.fit(Xtr, ytr)
                        m = metrics(y, model.predict(X))
                    except Exception as e:
                        m = {k: np.nan for k in METRICS}
                    rows.append(dict(set=set_name, image=fname, method=mname,
                                     condition=cond, lam=lam, **m))
            print(f"[{set_name}] {fi+1}/{len(files)} {fname}", end="\r")
    return pd.DataFrame(rows)

Corre el experimento. `quick=True` usa 3 imágenes/set (~30 s) para probar; `quick=False` corre las 200 (~15–30 min).

In [ ]:
df_ml = run_ml(quick=True)        # cambia a quick=False para la corrida completa
df_ml.to_csv(RESULTS / "results_ml.csv", index=False)
print("\nfilas:", len(df_ml))
df_ml.groupby(["method", "condition"])[METRICS].mean().round(3)

## 4. Agregación, test pareado y bootstrap

Para cada (set, método, métrica): media ± std de cada condición, Δ = media(BC) − media(sin BC),
IC95% bootstrap de Δ y p-valores (permutación pareada por sign-flip y Wilcoxon signed-rank).

In [ ]:
N_BOOT = 10000

def paired_perm_p(diff, n_perm=10000, rng=None):
    diff = diff[~np.isnan(diff)]
    if len(diff) == 0 or np.allclose(diff, 0):
        return np.nan
    obs = diff.mean()
    rng = rng or np.random.default_rng(SEED)
    perm = (rng.choice([-1, 1], size=(n_perm, len(diff))) * diff).mean(axis=1)
    return float((np.abs(perm) >= abs(obs) - 1e-12).mean())

def boot_ci(x, n_boot=N_BOOT, rng=None, alpha=0.05):
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = rng or np.random.default_rng(SEED)
    means = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    return tuple(np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)]))

def wilcoxon_p(bc, nobc):
    d = (bc - nobc); d = d[~np.isnan(d)]
    if len(d) == 0 or np.allclose(d, 0):
        return np.nan
    try:
        return float(stats.wilcoxon(bc, nobc).pvalue)
    except Exception:
        return np.nan

def build_summary(df, method_order):
    rng = np.random.default_rng(SEED)
    rows = []
    for s in SET_ORDER:
        for m in method_order:
            sub = df[(df["set"] == s) & (df["method"] == m)]
            if sub.empty:
                continue
            wide = sub.pivot_table(index="image", columns="condition", values=METRICS)
            for metric in METRICS:
                nobc = wide[(metric, "noBC")].to_numpy(); bc = wide[(metric, "BC")].to_numpy()
                diff = bc - nobc; lo, hi = boot_ci(diff, rng=rng)
                rows.append(dict(set=s, method=m, metric=metric, n=len(nobc),
                    noBC_mean=np.nanmean(nobc)*100, noBC_std=np.nanstd(nobc, ddof=1)*100,
                    BC_mean=np.nanmean(bc)*100, BC_std=np.nanstd(bc, ddof=1)*100,
                    diff=np.nanmean(diff)*100, ci_low=lo*100, ci_high=hi*100,
                    p_perm=paired_perm_p(diff, rng=rng), p_wilcoxon=wilcoxon_p(bc, nobc)))
    return pd.DataFrame(rows)

summary = build_summary(df_ml, METHOD_ORDER)
summary.to_csv(RESULTS / "summary_stats.csv", index=False)
summary.round(3).head(20)

### Tabla por set (formato media ± std, %)
`*` p<0.05, `**` p<0.01, `***` p<0.001 (test de permutación).

In [ ]:
def stars(p):
    return "" if pd.isna(p) else ("***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "")

def render_set(summary, s, metric="IoU"):
    sub = summary[(summary["set"] == s) & (summary["metric"] == metric)]
    out = []
    for _, r in sub.iterrows():
        out.append(dict(Método=r.method,
                        **{"Sin Box-Cox": f"{r.noBC_mean:.1f} ± {r.noBC_std:.1f}",
                           "Con Box-Cox": f"{r.BC_mean:.1f} ± {r.BC_std:.1f}",
                           "Δ (pp)": f"{r['diff']:+.1f}{stars(r.p_perm)}",
                           "IC95% Δ": f"[{r.ci_low:+.1f}, {r.ci_high:+.1f}]",
                           "p (perm)": f"{r.p_perm:.3f}"}))
    return pd.DataFrame(out)

for s in SET_ORDER:
    print(f"\n===== Set: {s} | métrica: IoU =====")
    display(render_set(summary, s, "IoU"))

### Exportar tabla Markdown (todas las métricas)

In [ ]:
def to_markdown(summary, method_order):
    lines = ["# Box-Cox vs sin Box-Cox — segmentación de grietas (media ± std %, 50 img/set)", ""]
    for s in SET_ORDER:
        lines += [f"## Set: {s}", "",
                  "| Método | Métrica | Sin Box-Cox | Con Box-Cox | Δ (pp) | IC95% Δ | p(perm) | p(Wilcoxon) |",
                  "|---|---|---|---|---|---|---|---|"]
        for m in method_order:
            for metric in METRICS:
                r = summary[(summary["set"]==s)&(summary["method"]==m)&(summary["metric"]==metric)]
                if r.empty: continue
                r = r.iloc[0]
                lines.append(f"| {m} | {metric} | {r.noBC_mean:.1f} ± {r.noBC_std:.1f} | "
                             f"{r.BC_mean:.1f} ± {r.BC_std:.1f} | {r['diff']:+.1f}{stars(r.p_perm)} | "
                             f"[{r.ci_low:+.1f}, {r.ci_high:+.1f}] | {r.p_perm:.3f} | {r.p_wilcoxon:.3f} |")
        lines.append("")
    return "\n".join(lines)

(RESULTS / "results_table.md").write_text(to_markdown(summary, METHOD_ORDER))
print("Guardado results_table.md")

## 5. (Opcional) Deep Learning — protocolo A vs C

U-Net, DeepLab y FCN. Compara **A** (entrenar+testear originales) vs **C**
(entrenar+testear Box-Cox).

> **Datos de entrenamiento.** Las 4 carpetas son el conjunto de *test* (50/set). Para entrenar
> sin contaminar el test hay dos opciones:
> 1. **Recomendado**: apuntar `TRAIN_RGB_DIR`/`TRAIN_MASK_DIR` a un conjunto externo con máscaras
>    (resto del dataset Ozgenel). El test sigue siendo los 4×50.
> 2. **Fallback**: dividir las 200 etiquetadas en train/test (estratificado por set). El test por
>    set será < 50; se indica con una advertencia.

Requiere `torch`, `torchvision`, `segmentation-models-pytorch` (celda 0).

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchvision.models.segmentation import fcn_resnet50

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
EPOCHS = 30
BATCH = 8
TRAIN_RGB_DIR = None    # p.ej. ROOT/"data"/"train_rgb"  (imágenes con máscara, fuera de los 200)
TRAIN_MASK_DIR = None    # p.ej. ROOT/"data"/"train_mask"
print("device:", DEVICE)

def list_labeled():
    items = []  # (rgb_path, mask_path, set_name)
    for s, folder in SETS.items():
        for f in sorted(os.listdir(MASK_DIR / folder)):
            if f.lower().endswith(".jpg") and (RGB_DIR / f).exists():
                items.append((RGB_DIR / f, MASK_DIR / folder / f, s))
    return items

class SegDS(Dataset):
    def __init__(self, items, use_boxcox):
        self.items, self.bc = items, use_boxcox
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        rgb_path, mask_path, _ = self.items[i]
        rgb, mask = load_pair(rgb_path, mask_path)
        feat, _ = make_feature(rgb, self.bc)            # [H,W] en [0,1]
        x = np.repeat(feat[None, :, :], 3, axis=0).astype(np.float32)  # 3 canales
        y = mask[None, :, :].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)

def make_model(name):
    if name == "U-Net":
        return smp.Unet("resnet34", in_channels=3, classes=1)
    if name == "DeepLab":
        return smp.DeepLabV3Plus("resnet34", in_channels=3, classes=1)
    if name == "FCN":
        m = fcn_resnet50(num_classes=1, weights=None); return m
    raise ValueError(name)

def forward_logits(model, x):
    out = model(x)
    return out["out"] if isinstance(out, dict) else out

def dice_bce_loss(logits, y):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    dice = 1 - (2*(p*y).sum() + 1) / (p.sum() + y.sum() + 1)
    return bce + dice

def train_model(name, train_items, use_boxcox):
    model = make_model(name).to(DEVICE)
    dl = DataLoader(SegDS(train_items, use_boxcox), batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for ep in range(EPOCHS):
        tot = 0
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = dice_bce_loss(forward_logits(model, x), y)
            loss.backward(); opt.step(); tot += loss.item()
        print(f"  {name} [{'BC' if use_boxcox else 'noBC'}] ep {ep+1}/{EPOCHS} loss {tot/len(dl):.3f}", end="\r")
    return model

@torch.no_grad()
def eval_model(model, test_items, use_boxcox, method):
    model.eval()
    rows = []
    for rgb_path, mask_path, s in test_items:
        rgb, mask = load_pair(rgb_path, mask_path)
        feat, _ = make_feature(rgb, use_boxcox)
        x = torch.from_numpy(np.repeat(feat[None, None], 3, axis=1).astype(np.float32)).to(DEVICE)
        pred = (torch.sigmoid(forward_logits(model, x))[0, 0].cpu().numpy() > 0.5).astype(np.uint8)
        rows.append(dict(set=s, image=rgb_path.name, method=method,
                         condition="BC" if use_boxcox else "noBC",
                         lam=np.nan, **metrics(mask.reshape(-1), pred.reshape(-1))))
    return rows

In [ ]:
# Construye train/test y corre A vs C para los 3 modelos.
from sklearn.model_selection import train_test_split

items = list_labeled()
if TRAIN_RGB_DIR and TRAIN_MASK_DIR:
    train_items = [(Path(TRAIN_RGB_DIR)/f, Path(TRAIN_MASK_DIR)/f, "train")
                   for f in sorted(os.listdir(TRAIN_MASK_DIR))]
    test_items = items                     # los 4×50 completos como test
    print(f"Train externo: {len(train_items)} | Test: {len(test_items)} (50/set)")
else:
    strat = [s for *_, s in items]
    tr, te = train_test_split(items, test_size=0.4, random_state=SEED, stratify=strat)
    train_items, test_items = tr, te
    print(f"[Fallback] Train {len(train_items)} / Test {len(test_items)} (test < 50/set)")

dl_rows = []
for name in ["U-Net", "DeepLab", "FCN"]:
    for use_bc in (False, True):           # A (False) vs C (True)
        model = train_model(name, train_items, use_bc)
        dl_rows += eval_model(model, test_items, use_bc, name)
    print()
df_dl = pd.DataFrame(dl_rows)
df_dl.to_csv(RESULTS / "results_dl.csv", index=False)
df_dl.groupby(["method", "condition"])[METRICS].mean().round(3)

### Tabla combinada ML + DL

In [ ]:
df_all = pd.concat([df_ml, df_dl], ignore_index=True)
summary_all = build_summary(df_all, METHOD_ORDER + ["U-Net", "DeepLab", "FCN"])
summary_all.to_csv(RESULTS / "summary_stats_all.csv", index=False)
(RESULTS / "results_table_all.md").write_text(to_markdown(summary_all, METHOD_ORDER + ["U-Net", "DeepLab", "FCN"]))
for s in SET_ORDER:
    print(f"\n===== Set: {s} | IoU (ML + DL) =====")
    display(render_set(summary_all, s, "IoU"))